# Klasifikasi multiclass / biner · toolkit modular v2

Class-specific bias dan scalar temperature berbeda tujuan; temperature tidak mengubah argmax.

Ubah konfigurasi. Hapus/tambah blok melalui perakit, bukan mengubah puluhan sel. Semua model menggunakan split/urutan label yang sama. Hasil harus dihitung ulang di Kaggle; skor notebook sumber tidak diwariskan.

Sumber adaptasi: `notebook/b_klasifikasi/10_multi_kelas/10_multi_kelas_v2_pretrained.ipynb`.

## 1 · Konfigurasi

File, kolom, preprocessing dan model hanya di sini. Group ID pengguna opsional. Untuk offline, set offline=True dan ganti model path ke bundle Kaggle input.

In [ ]:
CFG = {'data_file': '/kaggle/input/GANTI_DATASET/train.csv',
 'encoding': 'utf-8',
 'text_col': 'text',
 'label_col': 'label',
 'group_col': None,
 'seed': 42,
 'out': '/kaggle/working/hasil_toolkit',
 'offline': False,
 'preprocess': 'native',
 'slang': {},
 'stopwords': [],
 'eda_sample': 5000,
 'eda_top': 15,
 'eda_languages': False,
 'max_features': 60000,
 'max_len': 192,
 'epochs': 3,
 'learning_rate': 2e-05,
 'batch_size': 16,
 'e5_model': 'intfloat/multilingual-e5-base',
 'model_paths': {},
 'task': 'classification',
 'ensemble': True,
 'post': ['class_bias', 'temperature'],
 'models': ['dummy', 'tfidf_word', 'tfidf_union', 'word2vec', 'indobertweet'],
 'sentiment_pretrained': False}

## 2 · Lingkungan Kaggle

Pasang hanya dependensi komponen terpilih. Untuk bundle offline: rakit dengan --no-install atau hapus sel instalasi.

In [ ]:
import os, sys, subprocess
from pathlib import Path
assert Path('/kaggle/input').exists(), 'Jalankan di Kaggle.'
os.environ.update(USE_TF='0', USE_FLAX='0', TOKENIZERS_PARALLELISM='false')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'accelerate>=1,<2', 'datasets>=4,<5', 'emoji>=2,<3', 'ftfy>=6.3,<7', 'gensim>=4.4,<5', 'numpy', 'pandas', 'scikit-learn', 'scipy', 'sentencepiece>=0.2,<1', 'torch', 'transformers==5.20.2'], check=True)

## 3 · Fungsi komponen

Definisi sekali; sel pemakaian pendek berada di bawah. Semua dependensi disertakan, tanpa import dari folder toolkit.

### Kontrak, data loading & penyimpanan



In [ ]:
"""Kontrak bersama. Tidak memasang paket atau menjalankan model saat disalin."""
import json
import random
import re
import time
from pathlib import Path
import numpy as np
import pandas as pd

MODEL_RUNNERS = {}
ENCODERS = {}


def encode_e5(texts, cfg, role='query'):
    assert 'e5' in ENCODERS, 'Pasang komponen encoder E5 melalui perakit.'
    return ENCODERS['e5'](texts,cfg,role)


def register(name):
    def decorate(fn):
        MODEL_RUNNERS[name] = fn
        return fn
    return decorate


def read_table(path, encoding="utf-8", string_cols=()):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"Isi CFG['data_file'] dengan file Kaggle yang ada: {path}")
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    if path.suffix in {".jsonl", ".ndjson"}:
        return pd.read_json(path, lines=True)
    return pd.read_csv(path, sep="\t" if path.suffix == ".tsv" else ",", encoding=encoding,
        dtype={col:str for col in string_cols} or None)


def seed_all(seed=42):
    random.seed(seed)
    np.random.seed(seed)


def target_array(frame, cfg):
    return frame[cfg['label_cols']].to_numpy(float) if cfg['task']=='multilabel' else frame.y.to_numpy()


def require_kaggle(gpu=False):
    assert Path('/kaggle/input').exists(), "Jalankan eksperimen di Kaggle."
    if gpu:
        import torch
        assert torch.cuda.is_available(), "Aktifkan GPU Kaggle."


def show_table(name, table):
    from IPython.display import display
    print("\n" + name)
    display(table.to_frame() if isinstance(table, pd.Series) else table)


def save_tables(tables, out, prefix=""):
    out = Path(out)
    out.mkdir(parents=True, exist_ok=True)
    for name, table in tables.items():
        table.to_csv(out / f"{prefix}{name}.csv", index=True)


### Preprocessing native / normalized / BoW / IndoBERTweet

Slang hanya dari CFG["slang"]. Native mempertahankan negasi, emoji, angka, case. Uji normalisasi sebagai eksperimen.

In [ ]:
import html
import unicodedata

NEGATIONS = {'tidak', 'tak', 'bukan', 'belum', 'jangan', 'tanpa', 'gak', 'nggak', 'ga', 'not', 'never'}


def safe_text(value):
    """Missing tetap menjadi kosong; angka, emoji, negasi, tanda baca dipertahankan."""
    if not isinstance(value, str):
        return ''
    import ftfy
    text = ftfy.fix_text(value, unescape_html=False, uncurl_quotes=False,
        fix_latin_ligatures=False, fix_character_width=False, remove_control_chars=False, normalization=None)
    text = unicodedata.normalize('NFC', html.unescape(text))
    text = text.translate(dict.fromkeys(map(ord, '\ufeff\u200b\u2060\x00'), None))
    text = re.sub(r'[\x01-\x08\x0b\x0c\x0e-\x1f\x7f]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()


def preprocess_text(value, preset='native', slang=None, stopwords=()):
    text = safe_text(value)
    if preset == 'native':
        return text
    if preset == 'indobertweet':
        import emoji
        text = text.lower()
        text = re.sub(r'https?://\S+|www\.\S+|\burl\b', 'HTTPURL', text)
        text = re.sub(r'(?<!\w)@\w+|\buser\b', '@USER', text)
        return safe_text(emoji.demojize(text))
    if preset not in {'normalized', 'bow'}:
        raise ValueError(f'Preset tidak dikenal: {preset}')
    text = text.casefold()
    if slang:
        text = re.sub(r'\b[^\W_]+\b', lambda m: slang.get(m[0], m[0]), text)
    if preset == 'bow':
        text = re.sub(r'https?://\S+|www\.\S+|@\w+', ' ', text)
        # Stopword diberikan pengguna, negasi selalu dilindungi; tidak ada stemming otomatis.
        blocked = set(stopwords) - NEGATIONS
        text = ' '.join(w for w in re.findall(r'[^\W\d_]+', text) if w not in blocked)
    return safe_text(text)


def make_cleaner(cfg):
    return lambda value: preprocess_text(value, cfg['preprocess'], cfg.get('slang'), cfg.get('stopwords', ()))


### EDA angka: missing → vocabulary → noise

Basic/noise seluruh baris; vocab/ngram memakai sampel eksplisit. Persentase noise boleh overlap. Bahasa adalah estimasi sampel.

In [ ]:
from collections import Counter

NOISE_PATTERNS = {
    'url': r'https?://|www\.', 'mention': r'@\w+', 'hashtag': r'#\w+', 'angka': r'\d',
    'emoji_simbol': r'[\U0001F300-\U0001FAFF\u2600-\u27BF]', 'huruf_ulang': r'([^\W\d_])\1{2,}',
    'baca_ulang': r'[!?.,]{2,}', 'html': r'</?[a-zA-Z][^>]*>', 'encoding': r'\ufffd|Ã.|Â.',
    'non_ascii': r'[^\x00-\x7f]', 'spasi_ulang': r' {2,}', 'invisible': r'[\u200b\ufeff\u2060]',
    'kapital': r'(?-i:\b[A-Z]{3,}\b)', 'huruf_angka': r'\b(?=\w*[a-z])(?=\w*\d)\w+\b',
    'slang_contoh': r'\b(?:gk|gak|nggak|ga|bgt|yg|utk|udh|klo|jg)\b',
    'negasi': r'\b(?:tidak|tak|bukan|belum|jangan|tanpa|gak|nggak|ga|not|never)\b'
}


def numeric_eda(frame, text_col, label_cols=(), top=15, sample=5000, seed=42, languages=False):
    """Hitungan dasar seluruh baris; vocabulary/ngram pada sampel yang ukurannya dilaporkan."""
    s = frame[text_col]
    text = s.where(s.map(lambda x: isinstance(x, str)), '').fillna('')
    nonempty = text.str.strip().ne('')
    pseudo = text.str.strip().str.casefold().isin({'nan', 'none', 'null', 'n/a', 'na', '-', '--'})
    quality = pd.Series({'rows': len(s), 'missing': s.isna().sum(), 'blank': (~nonempty).sum(),
        'pseudo_null': pseudo.sum(), 'non_string_nonmissing': (s.notna() & ~s.map(lambda x: isinstance(x, str))).sum(),
        'unique_nonempty': text[nonempty].nunique(), 'duplicate_extra_nonempty': text[nonempty].duplicated().sum(),
        'duplicate_members_nonempty': text[nonempty].duplicated(keep=False).sum()})
    subset = text[nonempty].sample(min(int(sample), int(nonempty.sum())), random_state=seed)
    tokens = subset.str.casefold().str.findall(r'[^\W\d_]+')
    counts = pd.Series(Counter(w for row in tokens for w in row), dtype='int64').sort_values(ascending=False)
    n_tokens = int(counts.sum())
    tables = {'quality': quality.to_frame('n'),
        'missing_unique': pd.DataFrame({'missing': frame.isna().sum(), 'missing_pct': frame.isna().mean()*100,
            'unique': frame.apply(lambda col:col.map(lambda value:json.dumps(value,sort_keys=True) if isinstance(value,(list,dict)) else value).nunique())}),
        'length': pd.DataFrame({'char': text.str.len(), 'word': text.str.split().str.len()}).describe(percentiles=[.01,.5,.9,.95,.99]).T,
        'vocabulary': pd.Series({'sample_rows': len(subset), 'tokens': n_tokens, 'types': len(counts),
            'hapax_types': counts.eq(1).sum(), 'hapax_pct_types': counts.eq(1).mean()*100,
            'hapax_pct_tokens': counts.eq(1).sum()/max(n_tokens,1)*100}).to_frame('value'),
        'top_words': counts.head(top).to_frame('n').assign(pct_tokens=lambda d: d.n/max(n_tokens,1)*100)}
    for n in (2,3):
        grams = Counter(' '.join(row[i:i+n]) for row in tokens for i in range(len(row)-n+1))
        tables[f'ngram_{n}'] = pd.DataFrame(grams.most_common(top), columns=['term', 'n'])
    flags = pd.DataFrame({name: text.str.count(pattern, flags=re.I).gt(0) for name,pattern in NOISE_PATTERNS.items()})
    tables['noise'] = pd.DataFrame({'n_rows': flags.sum(), 'pct_all_rows_overlap_allowed': flags.mean()*100})
    tables['odd_words_candidates'] = counts[[len(w)>=20 or bool(re.search(r'([^\W\d_])\1{2,}',w)) for w in counts.index]].head(top).to_frame('n')
    for label in label_cols:
        known = frame[label].dropna()
        count = known.value_counts()
        tables[f'label_{label}'] = count.to_frame('n').assign(pct_known=lambda d:d.n/max(len(known),1)*100)
        tables[f'conflict_{label}'] = frame.assign(_key=text.str.casefold()).loc[nonempty].groupby('_key')[label].nunique().value_counts().to_frame('text_groups')
    if languages and len(subset):
        from langid.langid import LanguageIdentifier, model
        detector = LanguageIdentifier.from_modelstring(model, norm_probs=True)
        small = subset.sample(min(150,len(subset)),random_state=seed)
        pairs = small.map(lambda t: detector.classify(t[:3000]) if len(t.split())>=5 else ('uncertain',0))
        lang = pairs.map(lambda p:p[0] if p[1]>=.8 else 'uncertain').value_counts()
        tables['language_estimate_sample_only'] = lang.to_frame('n').assign(pct_sample=lang/len(small)*100)
    return tables


def multilabel_eda(frame, labels):
    y = frame[labels].astype(float)
    complete = y.dropna()
    co = pd.DataFrame(index=labels, columns=labels, dtype=float)
    for a in labels:
        for b in labels:
            known = y[[a,b]].notna().all(axis=1)
            co.loc[a,b] = (y.loc[known,a].eq(1)&y.loc[known,b].eq(1)).mean()*100
    return {'label_support': pd.DataFrame({'known':y.notna().sum(), 'unknown':y.isna().sum(),
                'positive':y.eq(1).sum(), 'negative':y.eq(0).sum(), 'positive_pct_known':y.mean()*100}),
        'cardinality_complete_rows': complete.sum(axis=1).value_counts().sort_index().to_frame('n'),
        'label_combinations_complete_rows': complete.value_counts().head(15).to_frame('n'),
        'cooccurrence_pct_pairwise_known': co}


def pair_eda(frame, a, b, target):
    left,right = frame[a].map(safe_text),frame[b].map(safe_text)
    pairs = pd.DataFrame({'a':left.str.casefold(),'b':right.str.casefold()})
    lset = pairs.a.map(lambda t:set(re.findall(r'[^\W\d_]+',t)))
    rset = pairs.b.map(lambda t:set(re.findall(r'[^\W\d_]+',t)))
    overlap = pd.Series([len(x&y)/max(len(x|y),1) for x,y in zip(lset,rset)])
    reversed_keys = set(zip(pairs.b,pairs.a))
    numbers = pd.Series([set(re.findall(r'\d+',x))==set(re.findall(r'\d+',y)) for x,y in zip(left,right)])
    detail = pd.DataFrame({'label_or_score':frame[target].to_numpy(),'jaccard':overlap,
        'abs_word_length_difference':(left.str.split().str.len()-right.str.split().str.len()).abs(),
        'identical':pairs.a.eq(pairs.b),'same_numbers':numbers})
    return {'pair_quality':pd.Series({'rows':len(frame),'identical_sides':pairs.a.eq(pairs.b).sum(),
            'duplicate_ordered_pairs_extra':pairs.duplicated().sum(),
            'pairs_with_reverse_present':sum((x,y) in reversed_keys for x,y in zip(pairs.a,pairs.b)),
            'shared_endpoints':len(set(pairs.a)&set(pairs.b))}).to_frame('n'),
        'pair_relation_statistics':detail.drop(columns='label_or_score').describe().T,
        'pair_relation_by_target':detail.groupby('label_or_score').agg(n=('jaccard','size'),mean_jaccard=('jaccard','mean'),
            mean_length_diff=('abs_word_length_difference','mean'),identical_pct=('identical',lambda v:v.mean()*100))}


def eda_for_task(frame, cfg):
    task = cfg['task'];tables={}
    columns = cfg['pair_cols'] if task in {'nli','similarity'} else [cfg.get('doc_text_col',cfg.get('text_col'))]
    labels = cfg['label_cols'] if task=='multilabel' else [cfg['label_col']] if task in {'classification','nli','similarity'} else []
    for i,col in enumerate(columns):
        part = numeric_eda(frame,col,labels if i==0 else (),cfg.get('eda_top',15),cfg['eda_sample'],cfg['seed'],cfg['eda_languages'])
        tables.update({str(col)+'_'+name:table for name,table in part.items()})
    if task in {'nli','similarity'}:
        tables.update(pair_eda(frame,*cfg['pair_cols'],cfg['label_col']))
    for name,table in tables.items():
        show_table(name,table)
    save_tables(tables,cfg['out'],'raw_eda_')
    return tables


### Metrik klasifikasi, partial multilabel & STS



In [ ]:
from sklearn.metrics import (f1_score, log_loss, classification_report, confusion_matrix,
    average_precision_score, precision_recall_fscore_support, mean_absolute_error, mean_squared_error)
from scipy.stats import pearsonr, spearmanr


def masked_multilabel_metrics(y, p, threshold=.5):
    y = np.asarray(y,float).copy()
    y[y<0] = np.nan
    p = np.asarray(p,float)
    assert p.shape==y.shape and np.isfinite(p).all()
    known = np.isfinite(y)
    pred = p>=threshold
    per, ap = [], []
    for j in range(y.shape[1]):
        k = known[:,j]
        per.append(f1_score(y[k,j],pred[k,j],zero_division=0) if k.any() else np.nan)
        ap.append(average_precision_score(y[k,j],p[k,j]) if np.any(y[k,j]==1) else np.nan)
    complete = known.all(axis=1)
    return {'macro_f1':float(np.nanmean(per)) if np.isfinite(per).any() else np.nan,
        'micro_f1':f1_score(y[known],pred[known],zero_division=0) if known.any() else np.nan,
        'macro_AP':float(np.nanmean(ap)) if np.isfinite(ap).any() else np.nan,
        'hamming_known':float(np.mean(y[known]!=pred[known])) if known.any() else np.nan,
        'exact_match_complete':float(np.all(y[complete]==pred[complete],axis=1).mean()) if complete.any() else np.nan,
        'n_complete':int(complete.sum()), 'n_known_cells':int(known.sum())}


def task_metrics(y, p, task):
    p = np.asarray(p)
    assert len(y)==len(p) and np.isfinite(p).all()
    if task=='similarity':
        y, p = np.asarray(y).ravel(), p.ravel()
        variable = len(y)>1 and np.std(y)>0 and np.std(p)>0
        return {'pearson':float(pearsonr(y,p).statistic) if variable else np.nan,
            'spearman':float(spearmanr(y,p).statistic) if variable else np.nan,
            'MAE':mean_absolute_error(y,p), 'RMSE':float(np.sqrt(mean_squared_error(y,p)))}
    if task=='multilabel':
        return masked_multilabel_metrics(y,p)
    assert p.ndim==2 and np.allclose(p.sum(1),1,atol=1e-4)
    pred = p.argmax(1)
    return {'macro_f1':f1_score(y,pred,labels=np.arange(p.shape[1]),average='macro',zero_division=0),
        'accuracy':float(np.mean(y==pred)), 'log_loss':log_loss(y,p,labels=np.arange(p.shape[1]))}


### Cross-fit threshold / bias / calibration / ensemble

Parameter memakai validation grup. Holdout tidak boleh memilih model/aturan. Duplicate consistency, audited rules, dan hierarki tersedia sebagai fungsi opsional.

In [ ]:
from sklearn.model_selection import GroupKFold
from scipy.special import softmax
from scipy.optimize import minimize_scalar


def fit_post(p, y, route, cfg):
    if route=='identity':
        return None
    if route=='isotonic':
        from sklearn.isotonic import IsotonicRegression
        return IsotonicRegression(out_of_bounds='clip').fit(p.ravel(),y)
    if route=='temperature':
        z = np.log(np.clip(p,1e-7,1))
        return float(minimize_scalar(lambda t:log_loss(y,softmax(z/t,axis=1),labels=np.arange(p.shape[1])),bounds=(.3,4),method='bounded').x)
    if route=='class_bias':
        bias = np.zeros(p.shape[1])
        for _ in range(2):
            for j in range(len(bias)):
                best,score = bias[j],-1
                for v in [-.8,-.4,0.,.4,.8]:
                    trial = bias.copy();trial[j]=v
                    s = task_metrics(y,apply_post(p,'class_bias',trial),cfg['task'])['macro_f1']
                    if (s,-abs(v))>(score,-abs(best)):
                        best,score = v,s
                bias[j] = best
        return bias
    if route in {'global_threshold','label_threshold'}:
        grid = [.03,.05,.1,.15,.2,.25,.3,.35,.4,.45,.5,.55,.6,.7,.8,.9]
        if route=='global_threshold':
            return max(grid,key=lambda c:(masked_multilabel_metrics(y,p,c)['macro_f1'],-abs(c-.5)))
        cuts = np.full(p.shape[1],.5)
        for j in range(p.shape[1]):
            k = np.isfinite(y[:,j]);target=y[k,j]
            if min(np.sum(target==1),np.sum(target==0))<cfg.get('threshold_min_support',5):
                continue
            cuts[j] = max(grid,key=lambda c:(f1_score(target,p[k,j]>=c,zero_division=0),-abs(c-.5)))
        return cuts
    raise ValueError(f'Post-processing tidak dikenal: {route}')


def apply_post(p, route, fitted):
    if route=='identity':
        return p.copy()
    if route=='isotonic':
        return fitted.predict(p.ravel())
    if route=='temperature':
        return softmax(np.log(np.clip(p,1e-7,1))/fitted,axis=1)
    if route=='class_bias':
        return softmax(np.log(np.clip(p,1e-7,1))+fitted,axis=1)
    if route in {'global_threshold','label_threshold'}:
        return (p>=fitted).astype(float)
    raise ValueError(route)


def select_policy(vp, y, groups, cfg):
    """Cross-fit parameter post di validation, kemudian refit; holdout tidak dibaca."""
    candidates = dict(vp)
    base = [name for name in vp if name!='dummy']
    if cfg.get('ensemble',False) and len(base)>1:
        candidates['ensemble'] = np.mean([vp[name] for name in base],axis=0)
    nfold = min(3,len(np.unique(groups)))
    routes = ['identity']+cfg.get('post',[])
    allowed = {'identity','isotonic'} if cfg['task']=='similarity' else {'identity','global_threshold','label_threshold'} if cfg['task']=='multilabel' else {'identity','temperature','class_bias'}
    assert set(routes)<=allowed, f'Post-processing tidak cocok dengan {cfg["task"]}: {set(routes)-allowed}'
    if nfold<2 and len(routes)>1:
        print('Post dinonaktifkan: validation kurang dari dua grup.')
        routes = ['identity']
    folds = list(GroupKFold(n_splits=nfold).split(y,groups=groups)) if nfold>=2 else []
    board = []
    for name,p in candidates.items():
        for route in routes:
            cv = p.copy()
            if route!='identity':
                for a,b in folds:
                    cv[b] = apply_post(p[b],route,fit_post(p[a],y[a],route,cfg))
            metrics = task_metrics(y,cv,cfg['task'])
            if cfg['task']=='multilabel':
                metrics['macro_AP'] = masked_multilabel_metrics(y,p)['macro_AP']
            board.append({'model':name,'route':route,**metrics})
    board = pd.DataFrame(board)
    metric,ascending = ('RMSE',True) if cfg['task']=='similarity' else ('macro_f1',False)
    # Identitas menang saat seri; temperature dapat dipilih untuk log-loss yang membaik dengan F1 sama.
    sort = [metric]+(['log_loss'] if 'log_loss' in board else [])
    board = board.sort_values(sort,ascending=[ascending]+([True] if len(sort)>1 else []),kind='stable')
    best = board.iloc[0]
    params = fit_post(candidates[best.model],y,best.route,cfg)
    return {'model':best.model,'route':best.route,'params':params,'members':base},board


def predict_policy(hp, policy):
    p = np.mean([hp[name] for name in policy['members']],axis=0) if policy['model']=='ensemble' else hp[policy['model']]
    return apply_post(p,policy['route'],policy['params']),p


def duplicate_consistency(probabilities, texts):
    """Opsional: rata-rata prediksi exact key dalam satu batch inference, tanpa label."""
    keys = pd.Series([safe_text(t).casefold() for t in texts])
    values = pd.DataFrame(probabilities)
    assert len(keys)==len(values)
    return values.groupby(keys).transform('mean').to_numpy()


def audited_rules(probabilities, texts, rules):
    """rules=[{'pattern':..., 'class_index':..., 'audited':True}]. Jangan belajar aturan dari holdout."""
    p = probabilities.copy()
    for rule in rules:
        assert rule.get('audited') is True, 'Aturan harus diaudit dari train dahulu.'
        for i,text in enumerate(texts):
            if re.search(rule['pattern'],text,flags=re.I):
                p[i] = 0
                p[i,rule['class_index']] = 1
    return p


def enforce_hierarchy(decisions, parent, children):
    """Opsional multilabel. Aktifkan hanya setelah audit anotasi train dan validasi grup."""
    result = decisions.astype(bool).copy()
    result[:,parent] |= result[:,children].any(axis=1)
    return result


### Grup pengguna, duplicate & connected endpoint



In [ ]:
from sklearn.model_selection import GroupShuffleSplit

def connected_groups(frame, text_cols, group_col=None, normalized_cols=(), shared_endpoints=False):
    """Gabung pengguna, duplikat native, dan collision normalisasi sebelum split."""
    parent = np.arange(len(frame))
    def root(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i
    seen = {}
    for i,(_,row) in enumerate(frame.iterrows()):
        keys = [(('endpoint' if shared_endpoints else col), safe_text(row[col]).casefold()) for col in text_cols]
        keys += [('endpoint' if shared_endpoints else 'normalized_'+col, str(row[col]).casefold()) for col in normalized_cols]
        if group_col and pd.notna(row[group_col]):
            keys.append(('user',str(row[group_col])))
        for key in keys:
            if not key[1]:
                continue
            if key in seen:
                parent[root(i)] = root(seen[key])
            else:
                seen[key] = i
    return np.array([root(i) for i in range(len(frame))])



### Split grup & partial multilabel

Kelompokkan pengguna + native duplicate + collision normalisasi. Pair memakai connected endpoints. NLI: e/n/c eksplisit. Unknown label tetap unknown.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit




def prepare_supervised(raw, cfg):
    cleaner = make_cleaner(cfg)
    frame = raw.copy().reset_index(drop=True)
    frame['_row_id'] = np.arange(len(frame))
    cols = cfg['pair_cols'] if cfg['task'] in {'nli','similarity'} else [cfg['text_col']]
    assert set(cols).issubset(frame), f'Kolom teks perlu {cols}; tersedia {frame.columns.tolist()}'
    for col, dest in zip(cols, ['a','b']):
        frame[dest] = frame[col].map(cleaner)
    valid = frame['a'].ne('')
    if len(cols)==2:
        valid &= frame.b.ne('')
    labels = cfg['label_cols'] if cfg['task']=='multilabel' else [cfg['label_col']]
    assert set(labels).issubset(frame), f'Kolom target perlu {labels}'
    if cfg['task']=='multilabel':
        for col in labels:
            values = pd.to_numeric(frame[col],errors='coerce')
            assert not (frame[col].notna() & values.isna()).any(), f'Label {col} tidak numerik'
            assert values.dropna().isin([0,1]).all(), f'Label {col} bukan 0/1/missing'
            frame[col] = values
        valid &= frame[labels].notna().any(axis=1)
    elif cfg['task']=='similarity':
        frame['y'] = pd.to_numeric(frame[cfg['label_col']],errors='coerce')
        valid &= np.isfinite(frame.y)
        assert frame.loc[valid,'y'].between(*cfg['score_range']).all(), 'Skor di luar score_range'
    else:
        valid &= frame[cfg['label_col']].notna()
        frame['label_name'] = frame[cfg['label_col']].astype(str).str.strip()
        if cfg['task']=='nli':
            mapping = cfg['nli_label_map']
            frame['label_name'] = frame.label_name.str.casefold().map(mapping)
            assert frame.loc[valid,'label_name'].notna().all(), 'Lengkapi nli_label_map untuk label data Anda'
    print('Baris invalid dibuang:',int((~valid).sum()))
    frame = frame.loc[valid].reset_index(drop=True)
    normalized = ['a','b'] if len(cols)==2 else ['a']
    frame['_group'] = connected_groups(frame, cols, cfg.get('group_col'), normalized, shared_endpoints=len(cols)==2)
    assert frame._group.nunique()>=5, 'Komponen grup terlalu sedikit; audit shared endpoint/duplikat/pengguna.'
    return frame


def split_supervised(frame, cfg):
    seed = cfg['seed']
    if cfg['task']=='multilabel':
        from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
        pos = frame.groupby('_group')[cfg['label_cols']].max().fillna(0)
        known = frame[cfg['label_cols']].notna().groupby(frame._group).max()
        ids, profile = pos.index.to_numpy(), np.hstack([pos, known.astype(int)])
        first = MultilabelStratifiedShuffleSplit(1,test_size=.2,random_state=seed)
        rest, test = next(first.split(ids,profile))
        fit, val = next(MultilabelStratifiedShuffleSplit(1,test_size=.25,random_state=seed+1).split(ids[rest],profile[rest]))
        partitions = [ids[rest][fit],ids[rest][val],ids[test]]
    else:
        rest, test = next(GroupShuffleSplit(1,test_size=.2,random_state=seed).split(frame,groups=frame._group))
        fit, val = next(GroupShuffleSplit(1,test_size=.25,random_state=seed+1).split(frame.iloc[rest],groups=frame.iloc[rest]._group))
        partitions = [frame.iloc[rest[fit]]._group.unique(),frame.iloc[rest[val]]._group.unique(),frame.iloc[test]._group.unique()]
    parts = {name:frame[frame._group.isin(ids)].reset_index(drop=True) for name,ids in zip(['train','val','test'],partitions)}
    if cfg['task'] in {'classification','nli'}:
        names = ['entailment','neutral','contradiction'] if cfg['task']=='nli' else sorted(parts['train'].label_name.unique())
        assert len(names)>=2, 'Train memerlukan minimal dua kelas'
        for name,part in parts.items():
            part['y'] = part.label_name.map({v:i for i,v in enumerate(names)})
            assert part.y.notna().all(), f'{name} mengandung kelas yang tidak ada di train; atur ulang split/grup.'
            part['y'] = part.y.astype(int)
        assert set(parts['train'].y)==set(range(len(names))), 'Train tidak mencakup semua kelas NLI'
    else:
        names = cfg['label_cols'] if cfg['task']=='multilabel' else ['score']
    parts['labels'] = list(names)
    for a,b in [('train','val'),('train','test'),('val','test')]:
        assert set(parts[a]._group).isdisjoint(parts[b]._group)
    print('Split baris:',{k:len(parts[k]) for k in ['train','val','test']}, '| label order:',names)
    return parts




### Support, FP/FN, confusion & slice



In [ ]:
"""Audit kesalahan; dipasang terpisah dari hitungan metrik/post-processing."""

def error_tables(frame, y, p, cfg, labels):
    task = cfg['task']
    rows = frame[['_row_id','a']+(['b'] if 'b' in frame else [])].copy()
    if task=='multilabel':
        known = np.isfinite(y)
        pred = p>=.5
        detail = []
        for j,label in enumerate(labels):
            k = known[:,j]
            pr,rc,f1,_ = precision_recall_fscore_support(y[k,j],pred[k,j],average='binary',zero_division=0) if k.any() else (np.nan,)*4
            detail.append({'label':label,'known':int(k.sum()),'positive':int(np.sum(y[k,j]==1)),
                'precision':pr,'recall':rc,'f1':f1,
                'FP':int(np.sum((y[k,j]==0)&pred[k,j])), 'FN':int(np.sum((y[k,j]==1)&~pred[k,j]))})
        rows['truth'] = [[labels[j] for j in range(len(labels)) if y[i,j]==1] for i in range(len(y))]
        rows['predicted'] = [[labels[j] for j in range(len(labels)) if pred[i,j]] for i in range(len(y))]
        rows['unknown'] = (~known).sum(1)
        rows['error_count'] = ((y!=pred)&known).sum(1)
        tables = {'per_label':pd.DataFrame(detail)}
    elif task=='similarity':
        rows['truth'],rows['predicted'] = y,p.ravel()
        rows['absolute_error'] = np.abs(y-p.ravel())
        tables = {}
    else:
        pred = p.argmax(1)
        rows['truth'] = [labels[i] for i in y]
        rows['predicted'] = [labels[i] for i in pred]
        rows['confidence'] = p.max(1)
        rows['error'] = y!=pred
        tables = {'per_class':pd.DataFrame(classification_report(y,pred,labels=np.arange(len(labels)),target_names=labels,output_dict=True,zero_division=0)).T,
            'confusion_counts':pd.DataFrame(confusion_matrix(y,pred,labels=np.arange(len(labels))),index=labels,columns=labels)}
    tables['errors'] = rows
    conditions = {'short_5_words':frame.a.str.split().str.len().le(5), 'long_40_words':frame.a.str.split().str.len().gt(40)}
    for name in ['negasi','emoji_simbol','slang_contoh']:
        conditions[name] = frame.a.str.count(NOISE_PATTERNS[name],flags=re.I).gt(0)
    slices = []
    for name,mask in conditions.items():
        k = mask.to_numpy()
        if k.any():
            row = task_metrics(y[k],p[k],task)
            if task=='multilabel':
                row.pop('macro_AP',None)  # Decisions after threshold are not probability AP.
            slices.append({'slice':name,'n':int(k.sum()),**row})
    tables['slices'] = pd.DataFrame(slices)
    return tables



### Jalankan model terpilih → freeze → audit holdout



In [ ]:
def run_supervised(data, cfg):
    """Model plug-in semuanya menghasilkan (validation, holdout) dengan urutan label yang sama."""
    seed_all(cfg['seed'])
    out = Path(cfg['out']);out.mkdir(parents=True,exist_ok=True)
    vp,hp,times = {},{},[]
    for name in cfg['models']:
        assert name in MODEL_RUNNERS, f'Blok model {name} belum disalin'
        start = time.perf_counter()
        vp[name],hp[name] = MODEL_RUNNERS[name](data,cfg)
        for split,p in [('val',vp[name]),('test',hp[name])]:
            assert len(p)==len(data[split]) and np.isfinite(p).all(), f'Output {name}/{split} invalid'
            if cfg['task']=='multilabel':
                assert p.shape==(len(data[split]),len(data['labels']))
        times.append({'model':name,'seconds':time.perf_counter()-start})
    yval = target_array(data['val'],cfg)
    policy,board = select_policy(vp,yval,data['val']._group.to_numpy(),cfg)
    show_table('Validation cross-fit (pengembangan, bukan test)',board)
    summary = {k:(v.tolist() if isinstance(v,np.ndarray) else v) for k,v in policy.items() if k!='params'}
    params = policy['params']
    if policy['route']=='isotonic':
        summary['params'] = {'x':params.X_thresholds_.tolist(),'y':params.y_thresholds_.tolist()}
    else:
        summary['params'] = params.tolist() if isinstance(params,np.ndarray) else params
    (out/'selection_frozen_before_test.json').write_text(json.dumps(summary,indent=2))
    # Baru sekarang baca target holdout. Tidak ada pemilihan ulang berdasarkan tabel berikut.
    ytest = target_array(data['test'],cfg)
    final,raw = predict_policy(hp,policy)
    rows = [{'model':name,**task_metrics(ytest,p,cfg['task'])} for name,p in hp.items()]
    score = task_metrics(ytest,final,cfg['task'])
    if cfg['task']=='multilabel':
        score['macro_AP'] = masked_multilabel_metrics(ytest,raw)['macro_AP']
    rows.append({'model':'FINAL',**score})
    tables = {'validation':board,'holdout':pd.DataFrame(rows),'timing':pd.DataFrame(times)}
    tables.update(error_tables(data['test'],ytest,final,cfg,data['labels']))
    if cfg['task']=='multilabel':
        for j,label in enumerate(data['labels']):
            k = np.isfinite(ytest[:,j])
            tables['per_label'].loc[j,'AP_raw_probability'] = average_precision_score(ytest[k,j],raw[k,j]) if np.any(ytest[k,j]==1) else np.nan
    save_tables(tables,out)
    np.savez_compressed(out/'predictions.npz',truth=ytest,probability=raw,decision_or_score=final,labels=np.array(data['labels']))
    pd.concat([data[k][['_row_id','_group']].assign(split=k) for k in ['train','val','test']]).to_csv(out/'split.csv',index=False)
    (out/'config.json').write_text(json.dumps(cfg,indent=2,default=str))
    show_table('Holdout · pilihan sudah dibekukan',tables['holdout'])
    show_table('Kelemahan per kondisi',tables['slices'])
    return {'policy':policy,'tables':tables,'validation_predictions':vp,'holdout_predictions':hp}


### Baseline prior / mean / semua-negatif



In [ ]:
@register('dummy')
def fit_dummy(data, cfg):
    y = target_array(data['train'],cfg)
    if cfg['task']=='multilabel':
        return tuple(np.zeros((len(data[s]),len(data['labels']))) for s in ['val','test'])
    if cfg['task']=='similarity':
        return tuple(np.full(len(data[s]),float(np.mean(y))) for s in ['val','test'])
    prior = np.bincount(y,minlength=len(data['labels']))/len(y)
    return tuple(np.tile(prior,(len(data[s]),1)) for s in ['val','test'])


### Head sparse/dense dan fitur pasangan



In [ ]:
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.dummy import DummyClassifier


def vectorizer(kind='word', features=60000):
    word = TfidfVectorizer(ngram_range=(1,2),min_df=1,max_features=features,sublinear_tf=True)
    if kind=='word':
        return word
    return FeatureUnion([('word',word),('char',TfidfVectorizer(analyzer='char_wb',ngram_range=(3,5),min_df=2,max_features=features,sublinear_tf=True))])


def pair_matrix(a, b, symmetric=False):
    delta = abs(a-b)
    return hstack([delta,a.multiply(b)] if symmetric else [a,b,delta,a.multiply(b)],format='csr')


def binary_heads(x, y, v, h, seed=42):
    vp,hp = np.zeros((v.shape[0],y.shape[1])),np.zeros((h.shape[0],y.shape[1]))
    for j in range(y.shape[1]):
        k = np.isfinite(y[:,j]);values = np.unique(y[k,j])
        if not len(values):
            print('Label tanpa known train, prior .5:',j)
            vp[:,j],hp[:,j] = .5,.5
            continue
        model = DummyClassifier(strategy='constant',constant=values[0]) if len(values)==1 else LogisticRegression(C=2,max_iter=1200,class_weight='balanced',random_state=seed)
        model.fit(x[k],y[k,j])
        if 1 in model.classes_:
            pos = list(model.classes_).index(1)
            vp[:,j],hp[:,j] = model.predict_proba(v)[:,pos],model.predict_proba(h)[:,pos]
    return vp,hp


def fit_head(x, v, h, data, cfg):
    y = target_array(data['train'],cfg)
    if cfg['task']=='multilabel':
        return binary_heads(x,y,v,h,cfg['seed'])
    if cfg['task']=='similarity':
        model = Ridge(alpha=5).fit(x,y)
        return tuple(np.clip(model.predict(m),*cfg['score_range']) for m in [v,h])
    model = LogisticRegression(C=2,max_iter=1200,class_weight='balanced',random_state=cfg['seed']).fit(x,y)
    assert np.array_equal(model.classes_,np.arange(len(data['labels'])))
    return model.predict_proba(v),model.predict_proba(h)


def fit_tfidf(data, cfg, kind):
    vec = vectorizer(kind,cfg.get('max_features',60000))
    pair = cfg['task'] in {'nli','similarity'}
    train = data['train']
    vec.fit(pd.concat([train.a,train.b]) if pair else train.a)
    matrices = []
    for name in ['train','val','test']:
        frame = data[name]
        a = vec.transform(frame.a)
        matrices.append(pair_matrix(a,vec.transform(frame.b),cfg['task']=='similarity') if pair else a)
    return fit_head(*matrices,data,cfg)


@register('tfidf_word')
def fit_word(data, cfg):
    return fit_tfidf(data,cfg,'word')


@register('tfidf_union')
def fit_word_char(data, cfg):
    return fit_tfidf(data,cfg,'union')


### Word2Vec train-only + LR / Ridge



In [ ]:
@register('word2vec')
def fit_word2vec(data, cfg):
    from gensim.models import Word2Vec
    pair = cfg['task'] in {'nli','similarity'}
    def tokenize(values):
        return [re.findall(r'[^\W\d_]+',str(t).casefold()) for t in values]
    train = data['train']
    corpus = tokenize(pd.concat([train.a,train.b]) if pair else train.a)
    model = Word2Vec(corpus,vector_size=100,min_count=1,window=5,workers=1,seed=cfg['seed'],epochs=10)
    def encode(values):
        out = []
        for tokens in tokenize(values):
            vectors = [model.wv[w] for w in tokens if w in model.wv]
            out.append(np.mean(vectors,axis=0) if vectors else np.zeros(100))
        return np.asarray(out)
    matrices = []
    for split in ['train','val','test']:
        f = data[split];a=encode(f.a)
        if pair:
            b = encode(f.b)
            a = np.hstack([np.abs(a-b),a*b]) if cfg['task']=='similarity' else np.hstack([a,b,np.abs(a-b),a*b])
        matrices.append(a)
    return fit_head(*matrices,data,cfg)


### Trainer classification / regression / masked BCE

Masked BCE untuk unknown; sigmoid multilabel. Model checkpoint hanya dari validation; NLI mempertahankan arah premise → hypothesis.

In [ ]:
PRETRAINED_MODELS = {
    'indobertweet':'indolem/indobertweet-base-uncased',
    'indobert_p1':'indobenchmark/indobert-base-p1',
    'xlmr':'FacebookAI/xlm-roberta-base',
    'indolem':'indolem/indobert-base-uncased',
    'nusabert':'LazarusNLP/NusaBERT-base'
}


def fit_pretrained(data, cfg, name):
    require_kaggle(gpu=True)
    import gc
    import torch
    from scipy.special import expit, softmax
    from datasets import Dataset, Value, Sequence
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding, set_seed
    set_seed(cfg['seed'])
    repo = cfg.get('model_paths',{}).get(name,PRETRAINED_MODELS[name])
    offline = cfg.get('offline',False)
    tok = AutoTokenizer.from_pretrained(repo,local_files_only=offline)
    task = cfg['task'];multi=task=='multilabel';reg=task=='similarity'
    problem = 'multi_label_classification' if multi else 'regression' if reg else 'single_label_classification'
    model = AutoModelForSequenceClassification.from_pretrained(repo,local_files_only=offline,
        num_labels=1 if reg else len(data['labels']),problem_type=problem,ignore_mismatched_sizes=True)
    if not reg:
        model.config.id2label = dict(enumerate(data['labels']))
        model.config.label2id = {name:i for i,name in enumerate(data['labels'])}
    def input_text(series):
        return series.map(lambda t:preprocess_text(t,'indobertweet')).tolist() if name=='indobertweet' else series.tolist()
    def encoded(frame, labels=True):
        a = input_text(frame.a)
        payload = tok(a,input_text(frame.b) if 'b' in frame else None,truncation=True,max_length=cfg['max_len'])
        if labels:
            y = target_array(frame,cfg)
            payload['labels'] = np.where(np.isfinite(y),y,-1).astype('float32') if multi else y.astype('float32' if reg else 'int64')
        ds = Dataset.from_dict(payload)
        if labels and reg:
            ds = ds.cast_column('labels',Value('float32'))
        if labels and multi:
            ds = ds.cast_column('labels',Sequence(Value('float32')))
        return ds
    y = target_array(data['train'],cfg)
    pos_weight = np.sqrt(np.clip(np.sum(y==0,axis=0)/np.maximum(np.sum(y==1,axis=0),1),1,20)) if multi else None
    class MaskedTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
            labels = inputs.pop('labels').float();mask=labels>=0
            output = model(**inputs)
            loss = torch.nn.functional.binary_cross_entropy_with_logits(output.logits.float(),labels.clamp(0,1),
                pos_weight=torch.as_tensor(pos_weight,device=labels.device,dtype=torch.float32),reduction='none')
            loss = (loss*mask).sum()/mask.sum().clamp(min=1)
            return (loss,output) if return_outputs else loss
    def probability(z):
        return z.ravel() if reg else expit(z) if multi else softmax(z,axis=1)
    def metric(prediction):
        score = task_metrics(prediction.label_ids,probability(prediction.predictions),task)
        return {'selection':-score['RMSE'] if reg else score['macro_f1']}
    out = Path(cfg['out'])/('model_'+name)
    args = TrainingArguments(output_dir=str(out),num_train_epochs=cfg['epochs'],learning_rate=cfg['learning_rate'],
        per_device_train_batch_size=cfg['batch_size'],per_device_eval_batch_size=cfg['batch_size']*2,
        gradient_accumulation_steps=cfg.get('accumulation',1),fp16=True,weight_decay=.01,
        eval_strategy='epoch',save_strategy='epoch',save_only_model=True,save_total_limit=1,
        load_best_model_at_end=True,metric_for_best_model='selection',greater_is_better=True,
        logging_strategy='epoch',disable_tqdm=True,report_to='none',seed=cfg['seed'])
    trainer_class = MaskedTrainer if multi else Trainer
    fit_frame = data['train']
    if reg:
        swapped = fit_frame.copy()
        swapped['a'],swapped['b'] = fit_frame.b,fit_frame.a
        fit_frame = pd.concat([fit_frame,swapped],ignore_index=True)
    trainer = trainer_class(model=model,args=args,train_dataset=encoded(fit_frame),eval_dataset=encoded(data['val']),
        data_collator=DataCollatorWithPadding(tok),compute_metrics=metric)
    if multi:
        trainer.model_accepts_loss_kwargs = False
    trainer.train()
    results = []
    for split in ['val','test']:
        frame = data[split]
        scores = probability(trainer.predict(encoded(frame,False)).predictions)
        if reg:
            swapped = frame.copy()
            swapped['a'],swapped['b'] = frame.b,frame.a
            scores = (scores+probability(trainer.predict(encoded(swapped,False)).predictions))/2
        results.append(scores)
    results = tuple(results)
    if reg:
        results = tuple(np.clip(p,*cfg['score_range']) for p in results)
    trainer.save_model(str(out));tok.save_pretrained(str(out))
    (out/'model_provenance.json').write_text(json.dumps({'repo':repo,'commit':getattr(model.config,'_commit_hash',None),'labels':data['labels']},indent=2))
    del trainer,model
    gc.collect();torch.cuda.empty_cache()
    return results


for _model_name in PRETRAINED_MODELS:
    MODEL_RUNNERS[_model_name] = lambda data,cfg,name=_model_name:fit_pretrained(data,cfg,name)


## 4 · Jalankan

Holdout dibuka setelah pemilihan validation dibekukan. Selisih kecil perlu repeated group split. EDA angka tersedia sebelum modeling.

In [ ]:
require_kaggle(gpu=True)
seed_all(CFG['seed'])

### Data & EDA raw

In [ ]:
raw = read_table(CFG['data_file'], CFG['encoding'])
tables = eda_for_task(raw, CFG)

### Preprocess & split

In [ ]:
prepared = prepare_supervised(raw, CFG)
data = split_supervised(prepared, CFG)
if CFG['task']=='multilabel':
    tables = multilabel_eda(data['train'], CFG['label_cols'])
    for name, table in tables.items(): show_table(name, table)
    save_tables(tables, CFG['out'], 'train_eda_')

### Model → post → evaluasi

In [ ]:
result = run_supervised(data, CFG)